In [18]:
import warnings

import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.api as sm
from scipy.stats import shapiro, levene
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('source/FReDA4.csv')
df2 = pd.read_csv('source/FReDA3.csv')
df["Group4"] = None

In [4]:
# satisfied = df[df["Group3"] == "Couple Satisfaction"].copy()
# mixed = df[df["Group3"] == "Couple Mixed"].copy()
# deprived = df[df["Group3"] == "Couple Deprivation"].copy()
# saturated = df[df["Group3"] == "Couple Oversaturation"].copy()
# deprived_one =  df[df["Group2"] == "One-sided Deprivation"].copy()
# deprived_both = df[df["Group1"] == "SubGroup3"].copy()
# deprived_me = df[df["Group1"] == "SubGroup2"].copy()
# deprived_partner = df[df["Group1"] == "SubGroup7"].copy()
# saturated_one =  df[df["Group2"] == "One-sided Oversaturation"].copy()
# saturated_both = df[df["Group1"] == "SubGroup6"].copy()
# saturated_me = df[df["Group1"] == "SubGroup5"].copy()
# saturated_partner = df[df["Group1"] == "SubGroup8"].copy()

In [5]:
# Satisfied
df.loc[df['Group3'] == 'Couple Satisfaction', 'Group4'] = 'Couple Satisfaction'

# Deprived groups
# df.loc[df['Group3'] == 'Couple Deprivation', 'Group4'] = 'Couple Deprivation'
# df.loc[df['Group2'] == 'One-sided Deprivation', 'Group4'] = 'Deprived_One'
df.loc[df['Group1'] == 'SubGroup-DeprivedBoth', 'Group4'] = 'Deprived_Both'
df.loc[df['Group1'] == 'SubGroup-DeprivedMe', 'Group4'] = 'Deprived_Me'
df.loc[df['Group1'] == 'SubGroup-DeprivedPartner', 'Group4'] = 'Deprived_Partner'
#
# Oversaturated groups
# df.loc[df['Group3'] == 'Couple Oversaturation', 'Group4'] = 'Couple Oversaturation'
# df.loc[df['Group2'] == 'One-sided Oversaturation', 'Group4'] = 'Oversaturated_One'
df.loc[df['Group1'] == 'SubGroup-OversaturatedBoth', 'Group4'] = 'Oversaturated_Both'
df.loc[df['Group1'] == 'SubGroup-OversaturatedMe', 'Group4'] = 'Oversaturated_Me'
df.loc[df['Group1'] == 'SubGroup-OversaturatedPartner', 'Group4'] = 'Oversaturated_Partner'

df.loc[df['Group3'] == 'Couple Mixed', 'Group4'] = 'Couple Mixed'


In [6]:
unique_values = df['Group4'].value_counts()
print(unique_values)

Group4
Couple Satisfaction      3842
Deprived_Both            3402
Deprived_Partner         2856
Deprived_Me              2084
Couple Mixed              660
Oversaturated_Partner     382
Oversaturated_Me          284
Oversaturated_Both         90
Name: count, dtype: int64


In [7]:
unique_values = df['Group3'].value_counts()
print(unique_values)

Group3
Couple Deprivation       8342
Couple Satisfaction      3842
Couple Oversaturation     756
Couple Mixed              660
Name: count, dtype: int64


In [61]:
df = df.rename(columns={
    'Self-esteem': 'Self_esteem',
    'Life Satisfaction': 'Life_Satisfaction',
    'Communication Quality': 'Communication_Quality',
    'Relationship Satisfaction': 'Relationship_Satisfaction',
    'Conflict Management': 'Conflict_Management'
})

In [62]:
traits = [
    'Neuroticism',
    'Extraversion',
    'Openness',
    'Agreeableness',
    'Conscientiousness',

    'Depressiveness',
    'Loneliness',
    'Self_esteem',
    'Life_Satisfaction',
    'Health',

    'Relationship_Satisfaction',
    'Communication_Quality',
    'Conflict_Management'
]

In [63]:
df = df.dropna(subset=traits).copy()

In [64]:
# -----------------------
# Assumption checks
# -----------------------
for trait in traits:
    print(f"\n--- {trait} ---")

    # Shapiro normality per group
    for group in df['Group4'].unique():
        data = df[df['Group4'] == group][trait]
        if len(data) >= 3:  # Shapiro requires >=3
            stat, p = shapiro(data)
            # print(f"{group} Shapiro p={p:.3f}")

    # Levene test for homogeneity of variance
    groups_data = [df[df['Group4'] == g][trait] for g in df['Group4'].unique()]
    stat, p = levene(*groups_data)
    print(f"Levene test p={p:.3f}")


--- Neuroticism ---
Levene test p=0.021

--- Extraversion ---
Levene test p=0.747

--- Openness ---
Levene test p=0.175

--- Agreeableness ---
Levene test p=0.007

--- Conscientiousness ---
Levene test p=0.613

--- Depressiveness ---
Levene test p=0.187

--- Loneliness ---
Levene test p=0.000

--- Self_esteem ---
Levene test p=0.000

--- Life_Satisfaction ---
Levene test p=0.016

--- Health ---
Levene test p=0.000

--- Relationship_Satisfaction ---
Levene test p=0.000

--- Communication_Quality ---
Levene test p=0.000

--- Conflict_Management ---
Levene test p=0.000


In [65]:
for trait in traits:
    print(f"\n=== ANOVA for {trait} ===")

    welch_anova = pg.welch_anova(dv=trait, between='Group4', data=df)
    # print(welch_anova)
    f_val = welch_anova.at[0, 'F']
    df_between = int(welch_anova.at[0, 'ddof1'])
    df_within = welch_anova.at[0, 'ddof2']  # Keep as float for Welch
    p_val = welch_anova.at[0, 'p-unc']
    eta_sq = welch_anova.at[0, 'np2']

    report = f"F({df_between}, {df_within:.2f}) = {f_val:.2f}; p = {p_val:.2E}; partial η2 = {eta_sq:.3f}"
    print(report)


=== ANOVA for Neuroticism ===
F(7, 911.30) = 22.99; p = 7.94E-29; partial η2 = 0.013

=== ANOVA for Extraversion ===
F(7, 912.14) = 4.55; p = 5.19E-05; partial η2 = 0.003

=== ANOVA for Openness ===
F(7, 916.11) = 6.24; p = 3.64E-07; partial η2 = 0.004

=== ANOVA for Agreeableness ===
F(7, 911.42) = 8.27; p = 8.38E-10; partial η2 = 0.005

=== ANOVA for Conscientiousness ===
F(7, 911.37) = 2.24; p = 2.95E-02; partial η2 = 0.001

=== ANOVA for Depressiveness ===
F(7, 912.41) = 23.42; p = 2.28E-29; partial η2 = 0.014

=== ANOVA for Loneliness ===
F(7, 908.30) = 60.03; p = 7.73E-71; partial η2 = 0.033

=== ANOVA for Self_esteem ===
F(7, 909.37) = 34.51; p = 8.44E-43; partial η2 = 0.019

=== ANOVA for Life_Satisfaction ===
F(7, 912.10) = 42.54; p = 4.58E-52; partial η2 = 0.024

=== ANOVA for Health ===
F(7, 911.04) = 24.99; p = 2.61E-31; partial η2 = 0.014

=== ANOVA for Relationship_Satisfaction ===
F(7, 912.74) = 184.93; p = 3.43E-170; partial η2 = 0.087

=== ANOVA for Communication_Qual

In [66]:
# # -----------------------
# # ANOVA + Effect size
# # -----------------------
# for trait in traits:
#     print(f"\n=== ANOVA for {trait} ===")
#
#     # OLS model
#     model = ols(f'{trait} ~ C(Group4)', data=df).fit()
#
#     # Standard ANOVA table
#     anova_table = sm.stats.anova_lm(model, typ=2)
#     print(anova_table)
#
#     # Eta squared from pingouin
#     eta2 = pg.anova(dv=trait, between='Group4', data=df,  effsize='n2', detailed=True)
#     print(f"Eta squared = {eta2.loc[0, 'n2']:.3f}")

In [67]:
print(df["Group4"].nunique())

8


In [68]:
# -----------------------
# Post-hoc tests
# -----------------------

traits = [
    'Neuroticism',
    'Extraversion',
    'Openness',
    'Agreeableness',
    'Conscientiousness',

    'Depressiveness',
    'Loneliness',
    'Self_esteem',
    'Life_Satisfaction',
    'Health',

    'Relationship_Satisfaction',
    'Communication_Quality',
    'Conflict_Management'
]

from pingouin import pairwise_tukey
all_results = []

for trait in traits:
    print(f"\n--- Post-hoc comparisons for {trait} ---")

    # Tukey HSD
    tukey = pairwise_tukey(
        dv=trait,
        between='Group4',
        data=df,
        effsize='cohen'
    )
    filter_turkey = tukey.loc[[0,7,8,9,10,11,12]].reset_index(drop=True)
    filter_turkey['Trait'] = trait
    all_results.append(filter_turkey)



--- Post-hoc comparisons for Neuroticism ---

--- Post-hoc comparisons for Extraversion ---

--- Post-hoc comparisons for Openness ---

--- Post-hoc comparisons for Agreeableness ---

--- Post-hoc comparisons for Conscientiousness ---

--- Post-hoc comparisons for Depressiveness ---

--- Post-hoc comparisons for Loneliness ---

--- Post-hoc comparisons for Self_esteem ---

--- Post-hoc comparisons for Life_Satisfaction ---

--- Post-hoc comparisons for Health ---

--- Post-hoc comparisons for Relationship_Satisfaction ---

--- Post-hoc comparisons for Communication_Quality ---

--- Post-hoc comparisons for Conflict_Management ---


In [69]:
final_turkey_4 = pd.concat(all_results, ignore_index=True)
# print(final_turkey)

In [70]:
# -----------------------
# Assumption checks
# -----------------------
for trait in traits:
    print(f"\n--- {trait} ---")

    # Shapiro normality per group
    for group in df['Group3'].unique():
        data = df[df['Group3'] == group][trait]
        if len(data) >= 3:  # Shapiro requires >=3
            stat, p = shapiro(data)
            # print(f"{group} Shapiro p={p:.3f}")

    # Levene test for homogeneity of variance
    groups_data = [df[df['Group3'] == g][trait] for g in df['Group3'].unique()]
    stat, p = levene(*groups_data)
    print(f"Levene test p={p:.3f}")


--- Neuroticism ---
Levene test p=0.007

--- Extraversion ---
Levene test p=0.283

--- Openness ---
Levene test p=0.478

--- Agreeableness ---
Levene test p=0.004

--- Conscientiousness ---
Levene test p=0.374

--- Depressiveness ---
Levene test p=0.048

--- Loneliness ---
Levene test p=0.000

--- Self_esteem ---
Levene test p=0.000

--- Life_Satisfaction ---
Levene test p=0.002

--- Health ---
Levene test p=0.000

--- Relationship_Satisfaction ---
Levene test p=0.000

--- Communication_Quality ---
Levene test p=0.000

--- Conflict_Management ---
Levene test p=0.000


In [71]:
for trait in traits:
    print(f"\n=== ANOVA for {trait} ===")

    welch_anova = pg.welch_anova(dv=trait, between='Group3', data=df)
    # print(welch_anova)
    f_val = welch_anova.at[0, 'F']
    df_between = int(welch_anova.at[0, 'ddof1'])
    df_within = welch_anova.at[0, 'ddof2']  # Keep as float for Welch
    p_val = welch_anova.at[0, 'p-unc']
    eta_sq = welch_anova.at[0, 'np2']

    report = f"F({df_between}, {df_within:.2f}) = {f_val:.2f}; p = {p_val:.2E}; partial η2 = {eta_sq:.3f}"
    print(report)


=== ANOVA for Neuroticism ===
F(3, 1558.61) = 44.01; p = 2.68E-27; partial η2 = 0.011

=== ANOVA for Extraversion ===
F(3, 1559.65) = 6.94; p = 1.22E-04; partial η2 = 0.002

=== ANOVA for Openness ===
F(3, 1567.40) = 6.88; p = 1.33E-04; partial η2 = 0.002

=== ANOVA for Agreeableness ===
F(3, 1554.79) = 10.51; p = 7.63E-07; partial η2 = 0.003

=== ANOVA for Conscientiousness ===
F(3, 1558.43) = 3.04; p = 2.80E-02; partial η2 = 0.001

=== ANOVA for Depressiveness ===
F(3, 1555.84) = 46.78; p = 6.05E-29; partial η2 = 0.011

=== ANOVA for Loneliness ===
F(3, 1551.15) = 117.63; p = 1.23E-68; partial η2 = 0.026

=== ANOVA for Self_esteem ===
F(3, 1554.93) = 70.47; p = 9.85E-43; partial η2 = 0.016

=== ANOVA for Life_Satisfaction ===
F(3, 1550.76) = 81.84; p = 3.75E-49; partial η2 = 0.020

=== ANOVA for Health ===
F(3, 1555.07) = 49.85; p = 9.24E-31; partial η2 = 0.012

=== ANOVA for Relationship_Satisfaction ===
F(3, 1551.08) = 370.93; p = 1.42E-181; partial η2 = 0.069

=== ANOVA for Commu

In [72]:
# -----------------------
# Post-hoc tests
# -----------------------

traits = [
    'Neuroticism',
    'Extraversion',
    'Openness',
    'Agreeableness',
    'Conscientiousness',

    'Depressiveness',
    'Loneliness',
    'Self_esteem',
    'Life_Satisfaction',
    'Health',

    'Relationship_Satisfaction',
    'Communication_Quality',
    'Conflict_Management'
]

from pingouin import pairwise_tukey
all_results = []

for trait in traits:
    print(f"\n--- Post-hoc comparisons for {trait} ---")

    # Tukey HSD
    tukey = pairwise_tukey(
        dv=trait,
        between='Group3',
        data=df,
        effsize='cohen'
    )

    filter_turkey = tukey.loc[[2,4,5]].reset_index(drop=True)
    filter_turkey['Trait'] = trait
    all_results.append(filter_turkey)


--- Post-hoc comparisons for Neuroticism ---

--- Post-hoc comparisons for Extraversion ---

--- Post-hoc comparisons for Openness ---

--- Post-hoc comparisons for Agreeableness ---

--- Post-hoc comparisons for Conscientiousness ---

--- Post-hoc comparisons for Depressiveness ---

--- Post-hoc comparisons for Loneliness ---

--- Post-hoc comparisons for Self_esteem ---

--- Post-hoc comparisons for Life_Satisfaction ---

--- Post-hoc comparisons for Health ---

--- Post-hoc comparisons for Relationship_Satisfaction ---

--- Post-hoc comparisons for Communication_Quality ---

--- Post-hoc comparisons for Conflict_Management ---


In [73]:
final_turkey = pd.concat(all_results, ignore_index=True)
# print(final_turkey)

In [74]:
dv = 'Neuroticism'
df.groupby("Group3")[dv].agg(["mean", "std"])

,mean,std
Group3,,
Couple Deprivation,8.084450,2.398884
Couple Mixed,8.395349,2.544806
Couple Oversaturation,7.908527,2.385777
Couple Satisfaction,7.566117,2.309391


In [75]:
summary = df.groupby('Group3')[traits].agg(['mean', 'std'])

final_table1 = summary.T.unstack(level=1)

group_order = ['Couple Satisfaction', 'Couple Deprivation', 'Couple Oversaturation', 'Couple Mixed']
final_table1 = final_table1.reindex(columns=group_order, level=0)

print(final_table1)

Group3                    Couple Satisfaction           Couple Deprivation  \
                                         mean       std               mean   
Neuroticism                          7.566117  2.309391           8.084450   
Extraversion                         9.397001  2.060889           9.255921   
Openness                            10.064468  2.243242           9.852619   
Agreeableness                       11.099850  1.941229          10.885099   
Conscientiousness                   11.085457  2.107962          10.982542   
Depressiveness                       4.794903  1.518108           5.124915   
Loneliness                           1.593703  0.876006           1.928813   
Self_esteem                         11.974213  2.251959          11.320612   
Life_Satisfaction                    8.229085  1.495542           7.763161   
Health                               3.917241  0.760679           3.734335   
Relationship_Satisfaction            9.319340  1.133408         

In [76]:
summary = df.groupby('Group4')[traits].agg(['mean', 'std'])

final_table2 = summary.T.unstack(level=1)
group_order = [
    'Deprived_Both',
    'Deprived_Me',
    'Deprived_Partner',
    'Oversaturated_Both',
    'Oversaturated_Me',
    'Oversaturated_Partner',
    'Couple Satisfaction',
    'Couple Mixed'
]

final_table2 = final_table2.reindex(columns=group_order, level=0)

print(final_table2)

Group4                    Deprived_Both           Deprived_Me            \
                                   mean       std        mean       std   
Neuroticism                    8.258096  2.373088    7.971998  2.433218   
Extraversion                   9.199934  2.029211    9.305331  2.049216   
Openness                       9.722406  2.245791    9.977921  2.178225   
Agreeableness                 10.763714  1.996696   11.024233  1.949236   
Conscientiousness             10.953734  2.133055   11.042542  2.096749   
Depressiveness                 5.230007  1.635951    5.071621  1.611428   
Loneliness                     2.056841  1.097589    1.880991  1.044974   
Self_esteem                   11.126239  2.486843   11.443726  2.465972   
Life_Satisfaction              7.604759  1.605702    7.883683  1.583671   
Health                         3.683410  0.797249    3.787830  0.783864   
Relationship_Satisfaction      8.140780  1.634345    8.614970  1.611800   
Communication_Quality    